# NiftyAlpha: Model Prototyping, Evaluation & XAI

## 1. Machine Learning Strategy & Objective
The central challenge in financial machine learning is distinguishing statistically meaningful predictive alpha from non-stationary market noise. In this research phase, we benchmark a linear baseline against a non-linear tree ensemble:

1. **Baseline Model**: L2-penalized Logistic Regression.
2. **Ensemble Architecture**: Random Forest Classifier with constrained depth to mitigate variance and prevent memorization of noise.

### The Precision-First Quantitative Imperative
In standard computer vision or NLP classification tasks, models are typically tuned to maximize balanced **Accuracy** or **ROC-AUC**. In quantitative trading, however, errors have asymmetric financial consequences:

$$\text{Precision} = \frac{\text{True Positives (Profitable Buys)}}{\text{True Positives} + \text{False Positives (Loss-Making Buys)}}$$

- **False Positive (Type I Error)**: The model flags a buy signal on an asset that subsequently depreciates. The portfolio incurs capital loss, bid-ask spread friction, exchange turnover fees, and statutory Indian Securities Transaction Tax (STT). **False positives consume real trading capital.**
- **False Negative (Type II Error)**: The model misses a profitable rally. While undesirable, the portfolio simply holds cash or accrues the 7% sovereign risk-free rate. **False negatives represent missed opportunities, not capital loss.**

Consequently, our quantitative objective is to optimize decision boundaries strictly for **Precision $\ge 60\%$** with non-zero recall.


In [ ]:
# Import modeling, metrics, and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Scikit-Learn Classifiers and Metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)

# Explainable AI (XAI)
import shap

warnings.filterwarnings('ignore')

# Visual style setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150


## 2. Temporal Splitting vs. Random Cross-Validation

### Why Standard K-Fold Cross-Validation Fails in Finance
In cross-sectional machine learning (e.g., customer churn, image classification), rows are assumed to be Independent and Identically Distributed (I.I.D.). Standard K-Fold splits observations randomly across folds.

In financial time-series, random shuffling is fatal:
1. **Lookahead Leakage**: Random splits place future observations ($t+10$) in the training set and past observations ($t$) in the test set, allowing the model to train on future market states.
2. **Serial Autocorrelation**: Financial returns exhibit temporal memory, volatility clustering (ARCH/GARCH effects), and macro regime transitions. Randomly sampling adjacent days bleeds identical market conditions between folds.

### Strict Chronological Partitioning
To guarantee complete isolation from lookahead bias, NiftyAlpha establishes strict temporal boundaries:
- **Training Set (Historical Ingestion)**: $\text{Date} \le \text{2021-12-31}$ (Captures multi-year bull, bear, and COVID recovery regimes).
- **Validation Set (Hyperparameter & Threshold Optimization)**: $\text{2022-01-01} \le \text{Date} \le \text{2023-12-31}$ (Used exclusively for probability calibration).
- **Test Set (Out-of-Sample Forward Simulation)**: $\text{Date} \ge \text{2024-01-01}$ (Completely untouched until Phase 3 backtesting).

> **Critical Scaling Rule**: Feature scaling (`StandardScaler`) must be fitted **strictly on the Training set**, then applied as a frozen transformation to Validation and Test sets.


In [ ]:
# Load pre-processed train and validation partitions
processed_dir = Path('../data/processed')

x_train_path = processed_dir / 'X_train.parquet'
y_train_path = processed_dir / 'y_train.parquet'
x_val_path = processed_dir / 'X_val.parquet'
y_val_path = processed_dir / 'y_val.parquet'

# Check if pre-processed files exist, otherwise generate synthetic research sample
if x_train_path.exists() and y_train_path.exists() and x_val_path.exists() and y_val_path.exists():
    print("Loading pre-processed parquet files from data/processed/...")
    X_train_raw = pd.read_parquet(x_train_path)
    y_train_raw = pd.read_parquet(y_train_path)
    X_val_raw = pd.read_parquet(x_val_path)
    y_val_raw = pd.read_parquet(y_val_path)
else:
    print("Pre-processed files not found; generating synthetic research partitions for demonstration...")
    np.random.seed(42)
    n_tr, n_val, n_f = 2000, 600, 15
    f_cols = [f'feature_{i}' for i in range(n_f)]
    
    tr_dates = pd.date_range('2018-01-01', periods=n_tr, freq='B')
    val_dates = pd.date_range('2022-01-01', periods=n_val, freq='B')
    
    X_train_raw = pd.DataFrame(np.random.randn(n_tr, n_f), columns=f_cols)
    X_train_raw.insert(0, 'Ticker', 'RELIANCE.NS')
    X_train_raw.insert(0, 'Date', tr_dates)
    
    y_train_raw = pd.DataFrame({
        'Date': tr_dates, 'Ticker': 'RELIANCE.NS',
        'is_positive_growth': (X_train_raw['feature_0'] + X_train_raw['feature_1']*0.5 + np.random.randn(n_tr) > 0.2).astype(int)
    })
    
    X_val_raw = pd.DataFrame(np.random.randn(n_val, n_f), columns=f_cols)
    X_val_raw.insert(0, 'Ticker', 'RELIANCE.NS')
    X_val_raw.insert(0, 'Date', val_dates)
    
    y_val_raw = pd.DataFrame({
        'Date': val_dates, 'Ticker': 'RELIANCE.NS',
        'is_positive_growth': (X_val_raw['feature_0'] + X_val_raw['feature_1']*0.5 + np.random.randn(n_val) > 0.2).astype(int)
    })

# Extract metadata columns (Date and Ticker) to retain auditability
meta_cols = ['Date', 'Ticker']
meta_train = X_train_raw[meta_cols].copy()
meta_val = X_val_raw[meta_cols].copy()

# Feature matrices excluding metadata
feature_cols = [c for c in X_train_raw.columns if c not in meta_cols]
X_train = X_train_raw[feature_cols].copy()
X_val = X_val_raw[feature_cols].copy()

# Target vectors (1D array for is_positive_growth)
y_train = y_train_raw['is_positive_growth'].astype(int).values
y_val = y_val_raw['is_positive_growth'].astype(int).values

print(f"\n[✓] Train Partition:      {X_train.shape[0]} rows x {X_train.shape[1]} features | Class 1 Ratio: {y_train.mean():.2%}")
print(f"[✓] Validation Partition: {X_val.shape[0]} rows x {X_val.shape[1]} features | Class 1 Ratio: {y_val.mean():.2%}")


In [ ]:
# Model Training: Baseline Logistic Regression vs. Random Forest Ensemble

# 1. Baseline Model: L2 Logistic Regression
baseline_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
baseline_model.fit(X_train, y_train)

# 2. Ensemble Model: Random Forest (Constrained Depth)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

print("=== Model Training Completed ===")
print(f"Baseline Logistic Regression Train Accuracy: {baseline_model.score(X_train, y_train):.4f}")
print(f"Random Forest Ensemble Train Accuracy:        {rf_model.score(X_train, y_train):.4f}")


## 3. Probability Threshold Calibration

### Shifting from Default 0.50 Threshold to High-Conviction Triggers
Off-the-shelf classifiers use an unweighted decision rule:
$$\hat{y} = \mathbb{I}(P(y=1|X) \ge 0.50)$$

In financial markets, a $51\%$ probability offers negligible statistical edge once execution slippage and taxes are factored in. Instead, we use model confidence scores $P(y=1|X)$ to sweep decision boundaries:
- Lower thresholds ($0.50$) produce high recall but low precision (frequent, noisy trades with high loss rates).
- Higher thresholds ($0.60 - 0.65$) act as **conviction filters**, discarding marginal setups and isolating trades with high probability of positive return.


In [ ]:
# Validation Evaluation: Precision Thresholding & PR-Curve

# Generate validation prediction probabilities for positive class (Buy)
y_val_prob = rf_model.predict_proba(X_val)[:, 1]

# Evaluate metrics across threshold range
threshold_sweep = [0.50, 0.55, 0.60, 0.65]
print("=========================================================================")
print(f"{'Threshold':<12} | {'Precision':<12} | {'Recall':<12} | {'F1-Score':<12} | {'Buy Signals'}")
print("=========================================================================")

for t in threshold_sweep:
    y_pred_t = (y_val_prob >= t).astype(int)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    f = f1_score(y_val, y_pred_t, zero_division=0)
    signals = np.sum(y_pred_t)
    print(f"  {t:<10.2f} | {p:<12.4f} | {r:<12.4f} | {f:<12.4f} | {signals:>5} / {len(y_val)}")

# Performance at selected 0.60 threshold
selected_t = 0.60
y_val_pred_target = (y_val_prob >= selected_t).astype(int)
target_prec = precision_score(y_val, y_val_pred_target, zero_division=0)
target_rec = recall_score(y_val, y_val_pred_target, zero_division=0)

# Precision-Recall Curve Plotting
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
pr_auc = average_precision_score(y_val, y_val_prob)

plt.figure(figsize=(10, 6))
plt.plot(recalls, precisions, color='#1f77b4', linewidth=2.5, label=f'Random Forest (PR-AUC = {pr_auc:.3f})')
plt.axhline(0.60, color='crimson', linestyle='--', label='Target Precision Benchmark (60%)')
plt.scatter([target_rec], [target_prec], color='crimson', s=120, zorder=5, label=f'Chosen Operating Point (T = {selected_t:.2f})')

plt.title('Precision-Recall Curve (Validation Set: 2022-2023)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Recall (Fraction of Growth Opportunities Captured)', fontsize=12)
plt.ylabel('Precision (Fraction of Buy Signals that were Profitable)', fontsize=12)
plt.legend(loc='lower left', frameon=True, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 4. Explainable AI (XAI) via SHAP

### Moving Beyond Black-Box Trading
Institutional risk management and compliance require transparent rationale for automated trade signals. We employ **SHAP (SHapley Additive exPlanations)**, derived from cooperative game theory, which guarantees:
1. **Local Accuracy**: The sum of feature contributions equals the difference between model output and expected base rate.
2. **Consistency**: If a model changes so that a feature has a larger marginal impact, that feature's attribution strictly increases.

We deploy `shap.TreeExplainer`, which computes exact Shapley attributions in polynomial time $\mathcal{O}(TLD^2)$, making it orders of magnitude faster and more reliable than sampling-based kernel approximations.


In [ ]:
# SHAP Interpretability: Global Feature Attributions

# Sample 200 validation observations for responsive computation
sample_size = min(200, len(X_val))
X_val_sample = X_val.sample(sample_size, random_state=42)

print(f"Calculating SHAP TreeExplainer attributions across {sample_size} validation samples...")

try:
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_val_sample)

    # Focus on Class 1 (Buy signal attributions)
    if isinstance(shap_values, list) and len(shap_values) == 2:
        vals_to_plot = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        vals_to_plot = shap_values[:, :, 1]
    else:
        vals_to_plot = shap_values

    plt.figure(figsize=(10, 8))
    shap.summary_plot(vals_to_plot, X_val_sample, plot_type='dot', show=False)
    plt.title('SHAP Value Distribution: Feature Drivers for Buy Signals (Class 1)', fontsize=13, fontweight='bold', pad=14)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"SHAP runtime exception: {e}. Rendering native Gini feature importances:")
    importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 6))
    importances.plot(kind='barh', color='#00CC96')
    plt.gca().invert_yaxis()
    plt.title('Top 15 Predictive Features (Random Forest Gini Importance)', fontsize=13, fontweight='bold')
    plt.xlabel('Relative Feature Importance')
    plt.tight_layout()
    plt.show()
